# Projeto Final – Análise da População

Este notebook organiza uma análise dos dados do Censo Demográfico de 2022, comparando a população de 2010 e 2022. A análise começa pela preparação e conferência da base, segue para os resultados por estado e, por fim, detalha os municípios.

## 1. Carregamento da base

Primeiro, a planilha original é importada e são mantidos apenas os registros de municípios identificados corretamente.

In [ ]:
import pandas as pd

arquivo = "CD2022_Populacao_2010_Compatibilizada_20231222.xlsx"
dados = pd.read_excel(arquivo, sheet_name="Municípios", header=2)
dados = dados.dropna(subset=["UF", "NOME DO MUNICÍPIO"]).copy()

print(f"Registros carregados: {len(dados):,}")
dados.head()

## 2. Preparação e validação dos dados

As colunas de população são convertidas para formato numérico. Em seguida, é calculada a diferença absoluta entre 2022 e 2010, que será usada para ordenar os resultados.

In [ ]:
coluna_2010 = "População 2010 (Alterações de Limites até 2022)1"
coluna_2022 = "População Censo 2022"

for coluna in (coluna_2010, coluna_2022):
    dados[coluna] = pd.to_numeric(dados[coluna], errors="coerce")

dados["DIFERENCA"] = dados[coluna_2022] - dados[coluna_2010]

print("Valores ausentes nas populações:")
print(dados[[coluna_2010, coluna_2022]].isna().sum())

dados[["UF", "NOME DO MUNICÍPIO", coluna_2010, coluna_2022, "DIFERENCA"]].head()

## 3. Visão geral da variação

Antes de separar os dados por região, observamos quantos municípios apresentaram aumento ou redução da população no período.

In [ ]:
aumentaram = (dados["DIFERENCA"] > 0).sum()
reduziram = (dados["DIFERENCA"] < 0).sum()
sem_variacao = (dados["DIFERENCA"] == 0).sum()

resumo = pd.DataFrame({
    "Situação": ["Aumento", "Redução", "Sem variação"],
    "Quantidade de municípios": [aumentaram, reduziram, sem_variacao]
})

resumo

## 4. População agregada por estado

A população municipal é somada por UF. Depois, a diferença entre os dois censos permite identificar os estados que tiveram os maiores aumentos ou reduções em valores absolutos.

In [ ]:
estado = (
    dados.groupby("UF", as_index=False)[[coluna_2010, coluna_2022]]
    .sum()
)

estado["DIFERENCA"] = estado[coluna_2022] - estado[coluna_2010]
estado = estado.sort_values("DIFERENCA", ascending=False).reset_index(drop=True)

estado

### Destaques estaduais

In [ ]:
print("5 estados com maior aumento líquido:")
display(estado.head(5))

print("5 estados com maior redução líquida:")
display(estado.tail(5).sort_values("DIFERENCA"))

## 5. Detalhamento por município

Agora a mesma comparação é mantida no nível municipal, preservando os códigos de identificação da UF e do município.

In [ ]:
municipio = dados[[
    "UF",
    "COD. UF",
    "COD. MUNIC",
    "NOME DO MUNICÍPIO",
    coluna_2010,
    coluna_2022,
    "DIFERENCA"
]].copy()

municipio = municipio.sort_values("DIFERENCA", ascending=False).reset_index(drop=True)

municipio.head(20)

### Maiores aumentos e maiores reduções

In [ ]:
print("10 municípios com maior aumento:")
display(municipio.head(10))

print("10 municípios com maior redução:")
display(municipio.tail(10).sort_values("DIFERENCA"))

## 6. Exportação dos resultados

As tabelas finais são exportadas em CSV para que possam ser utilizadas posteriormente em planilhas ou outros programas de análise.

In [ ]:
estado.to_csv(
    "populacao_por_estado.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

municipio.to_csv(
    "populacao_por_municipio.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos populacao_por_estado.csv e populacao_por_municipio.csv foram atualizados.")

## 7. Conclusão

A análise permite comparar a evolução populacional entre 2010 e 2022 em duas escalas: estadual e municipal. A ordenação pela diferença absoluta facilita a identificação dos locais que mais aumentaram ou reduziram sua população no período.